# Cuaderno de la sesión 1. Ver el vector antes de guardarlo

En este cuaderno no hay base de datos. El objetivo es obtener un vector y mirarlo.

Un modelo de embeddings ya entrenado convierte una frase en una lista de números. Esa lista es el vector: la posición de la frase en un mapa de muchas dimensiones. No es un resumen y no es una traducción. No se lee número a número.

El modelo de esta sesión es pequeño y gratuito (`all-MiniLM-L6-v2`). Devuelve **384** números porque así se construyó. En DocIA+ el modelo del proyecto será Amazon Titan y devolverá **1024**. Son dos mapas distintos. No se mezclan.

Las frases son de ejercicio y ya son cortas: cada una es el trozo. No son documentos oficiales del IES.

La primera ejecución descarga el modelo. Puede tardar un minuto. No lo estamos entrenando; lo estamos usando.


## Instalar

Python no trae un modelo de embeddings. Esta línea descarga la librería que sabe cargar uno.

`sentence-transformers` es el programa. El modelo, `all-MiniLM-L6-v2`, se bajará en la celda siguiente, la primera vez que se use. Sin esta librería no hay función que convierta texto en números.


In [ ]:
!pip install sentence-transformers -q
print("Libreria instalada")


## Convertir una frase en una lista de números

`SentenceTransformer(...)` carga el modelo en memoria. `encode` es el embedding: entra la frase y sale la lista.

Imprimimos el texto, la longitud y solo los diez primeros números. La longitud tiene que ser 384. Los diez primeros bastan para ver que son decimales, no palabras. El número 7, solo, no significa nada. El significado, si lo hay, aparece al comparar la lista completa con otra lista del mismo modelo.

El modelo no redacta y no contesta a la frase. Solo la coloca en el mapa.


In [ ]:
from sentence_transformers import SentenceTransformer

modelo = SentenceTransformer("all-MiniLM-L6-v2")

frase = "Procedimiento de formalizacion de matricula en el curso de especializacion."
vector = modelo.encode(frase)

print("Texto:")
print(frase)
print()
print("Cuantos numeros tiene el vector:", len(vector))
print("Los 10 primeros:")
print(vector[:10])


## Medir quién está cerca

La cercanía es una propiedad de los vectores. No hace falta una base de datos para verla.

Se convierten tres frases con el mismo modelo. Python resta dos listas y mide lo larga que queda esa resta. Eso es una distancia. Número más pequeño, más cerca.

Esta cuenta es la longitud de la resta entre dos listas. No es el número que imprime ChromaDB en el cuaderno siguiente. Allí, con el espacio coseno, la distancia es 1 menos el coseno. Las dos se leen igual (más pequeña, más cerca) y no se comparan entre sí: un 0,8 aquí no es «peor» que un 0,002 allí.

Tiene que quedar más cerca «matrícula» de «inscripción» que de «cafetería». Las dos primeras hablan del mismo trámite con otras palabras. La tercera no. Si saliera al revés, las tres frases no se habrían codificado con el mismo modelo.


In [ ]:
import numpy as np

frases = {
    "matricula": "Procedimiento de formalizacion de matricula en el curso de especializacion.",
    "inscripcion": "Instrucciones para inscribirse en el curso.",
    "cafeteria": "Menu diario de la cafeteria del instituto.",
}
vectores = {nombre: modelo.encode(texto) for nombre, texto in frases.items()}

cerca = np.linalg.norm(vectores["matricula"] - vectores["inscripcion"])
lejos = np.linalg.norm(vectores["matricula"] - vectores["cafeteria"])

print(f"Distancia matricula - inscripcion: {cerca:.4f}")
print(f"Distancia matricula - cafeteria:   {lejos:.4f}")
print()
if cerca < lejos:
    print("La inscripcion queda mas cerca de la matricula que la cafeteria.")
else:
    print("El resultado no cuadra: revisa que las tres frases se han codificado con el mismo modelo.")
